# 02 - CWRU Feature Engineering

Convert raw CWRU vibration signals into the canonical **MachineFeatureVector** by slicing each recording into 2048-sample windows and computing time-domain descriptors.

Pipeline:

```
raw .mat  ->  drive-end signal  ->  2048-sample windows  ->  feature rows  ->  pandas DataFrame
```

The implementation lives in `ml/src/feature_extraction.py` and `ml/src/dataset_builder.py` so this notebook is a thin orchestration layer.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'ml').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

RAW_DIR = PROJECT_ROOT / 'ml' / 'data' / 'raw' / 'cwru'
PROCESSED_DIR = PROJECT_ROOT / 'ml' / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
from ml.src.dataset_builder import (
    CWRU_RECORDINGS,
    build_feature_frame,
    resolve_recording_paths,
)

resolved, missing = resolve_recording_paths(RAW_DIR)
print(f'Resolved recordings: {len(resolved)} / {len(CWRU_RECORDINGS)}')
if missing:
    print('Missing recordings (drop the .mat files into ml/data/raw/cwru/ to include them):')
    for spec in missing:
        print(f"  - {spec.recording_id}: expected one of {spec.candidate_filenames}")

In [ ]:
if not resolved:
    raise SystemExit('No CWRU recordings found - see 01_cwru_exploration.ipynb for download instructions.')

features_df = build_feature_frame(RAW_DIR)
features_df.head()

In [ ]:
print('rows per recording:')
print(features_df.groupby('recording_id').size())
print()
print('rows per fault class:')
print(features_df['fault_class'].value_counts())

In [ ]:
output_path = PROCESSED_DIR / 'cwru_features.parquet'
features_df.to_parquet(output_path, index=False)
print(f'wrote {len(features_df)} rows to {output_path}')